# XFakeNet – grid search and hyperparameter tuning

Tunes the main model (XFakeNet v2, augmented corpus) with a full grid search, chosen on the **validation set
only**, then retrains the best setting with 3 seeds and compares it with the default settings on the test set
and under a fresh TextFooler attack.

| Section | What it does | Time on T4 |
|---|---|---|
| A | **Grid search**: encoder learning rate {1e-5, 2e-5, 3e-5} × batch size {16, 32} × dropout {0.1, 0.3}, each up to 4 epochs = 12 runs (48 setting × epoch points) | ~2.5–3 h |
| B | **Best setting, 3 seeds** + TextFooler attack on each | ~45 min |
| C | **Results** (tables + heatmap), saved to Drive and downloaded | seconds |

**Free Colab may stop you after a few hours (GPU limit).** That's fine: every finished run is saved to Drive.
If you get disconnected or see "Cannot connect to GPU", come back later (or the next day) and `Run all` again;
it continues where it stopped.

**Steps:** `Runtime → Change runtime type → T4 GPU`, `Runtime → Run all`, allow Google Drive, keep this tab open.

## 1. GPU, Google Drive, and earlier results

In [ ]:
import os, shutil, subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU (or try again later if Colab's GPU limit was hit).")
print("GPU:", gpu)
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/xfakenet"
os.chdir(WORK)
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Google Drive: {free/1e9:.1f} GB free")
need = ["data/processed/corpus_aug_train.csv", "data/processed/corpus_valid.csv", "data/processed/corpus_test.csv",
        "runs/xfakenet_v2/metrics.json", "data/augment/textfooler_test_xfakenet_v2.jsonl"]
missing = [f for f in need if not os.path.exists(f)]
if missing:
    raise SystemExit(f"Missing files: {missing}")
print("Data and earlier results found.")

## 2. Write the project code

In [ ]:
import os
for d in ['configs','src/xfakenet','scripts','results/grid','results/final']:
    os.makedirs(d, exist_ok=True)
print('folders ready')

In [ ]:
%%writefile configs/week1.yaml
seed: 42
paths:
  raw: data/raw
  processed: data/processed
  images: data/images
  runs: runs

liar:
  dir: data/raw/liar            # expects train.tsv / valid.tsv / test.tsv

extra_classes:
  enabled: false      # three-class setup: LIAR only
  source: tshp17
  file: data/raw/tshp17/fulltrain.csv   # columns: label,text (no header)
  test_file: data/raw/tshp17/balancedtest.csv
  use_hoax_trusted: false
  # Articles are long; LIAR statements are ~18 words. Truncate to the first
  # sentences up to this many words so length is not a shortcut for the class.
  max_words: 40
  # Cap per class so Satire/Propaganda stay minority classes (as in the outline).
  max_per_class: 1500

fakenewsnet:
  dir: data/raw/FakeNewsNet/dataset
  sample_size: 1000            # articles to try for images (balanced fake/real)
  min_image_side: 100          # px; smaller images (icons, pixels) are discarded
  workers: 16
  timeout: 10
  use_wayback: true

baseline:
  model: roberta-base
  max_len: 64
  epochs: 4
  batch_size: 32
  lr: 2.0e-5
  warmup_ratio: 0.06
  weight_decay: 0.01
  fp16: true

# ---------------- Week 2 ----------------
augmentation:                  # final training mix (week 3): 70% original / 15% / 15%
  original_share: 70
  textfooler_share: 15
  paraphrase_share: 15

cscore:
  clip_model: openai/clip-vit-base-patch32
  batch_size: 64
  percentile: 95               # alternative threshold: 95th percentile of aligned C-scores (valid)
  out_dir: results/cscore

paraphrase:
  models: [gemini-3.5-flash-lite, gemini-3.8-flash, gemini-3.1-flash-lite, gemini-2.5-flash]
  batch_size: 20               # statements per API call
  temperature: 0.9
  sleep_s: 4                   # pause between calls (free-tier rate limits)
  test_n: 500
  out_train: data/augment/paraphrase_train.jsonl
  out_test: data/augment/paraphrase_test.jsonl

textfooler:
  victim_dir: runs/baseline/best
  embeddings_url: https://github.com/nmrksic/counter-fitting/raw/master/word_vectors/counter-fitted-vectors.txt.zip
  embeddings_path: data/raw/counter-fitted-vectors.txt.zip
  sim_model: sentence-transformers/all-MiniLM-L6-v2
  k: 50                        # synonyms per word (paper: 50)
  sim_threshold: 0.80
  max_change: 0.25             # at most 25% of words replaced
  query_budget: 2000
  test_n: 500
  out_train: data/augment/textfooler_train.jsonl
  out_test: data/augment/textfooler_test.jsonl

model_v1:
  lrs: [1.0e-5, 2.0e-5, 3.0e-5]
  epochs: 5

# ---------------- Week 3 ----------------
xfakenet:                      # RoBERTa + jointly trained token/segment attention
  encoder: roberta-base
  max_len: 64
  max_segments: 8
  att_dim: 256
  epochs: 4
  batch_size: 32
  lr: 2.0e-5                   # encoder
  head_lr: 1.0e-4              # attention layers + classifier
  warmup_ratio: 0.06
  weight_decay: 0.01
  highlight_threshold: 0.65    # outline: tokens above T = 0.65 are highlighted

modality:
  C: 0.1                       # regularisation of the text/image logistic models (389 items)

week3:
  out_dir: results/week3

# ---------------- Week 4 ----------------
week4:
  out_dir: results/week4

deception:
  n_mismatched: 200            # shuffled pairs added to the 389 real ones for labelling
  batch_size: 5                # image-text pairs per Gemini call
  image_side: 512
  sleep_s: 4
  C: 0.5
  min_per_class: 5
  handcheck_n: 100
  labels: data/deception/gemini_labels.jsonl
  features: data/deception/clip_features.npz
  oof: data/deception/classifier_oof.csv
  model: runs/deception_clf.joblib
  handcheck_sample: data/deception/handcheck_sample.csv
  handcheck_results: data/deception/handcheck_results.csv

pipeline:
  modality_model: runs/modality.joblib

explain:                       # faithful explanations (week 4 fix)
  permutations_gpu: 10
  permutations_cpu: 6
  cpu_statements: 400
  chunk: 50

# ---------------- Week 5: grid search (mentor's request) ----------------
grid:
  lrs: [1.0e-5, 2.0e-5, 3.0e-5]      # encoder learning rate
  batch_sizes: [16, 32]
  dropouts: [0.1, 0.3]
  max_epochs: 4                       # validation macro-F1 is recorded after every epoch
  final_seeds: [42, 1, 2]
  out_dir: results/grid

In [ ]:
%%writefile configs/label_mapping.yaml
# XFakeNet five-class label mapping (frozen 2026-10-21, Week 1)
# Every later step (training, augmentation, evaluation) reads this file.
# Change it only with a note in docs/label_mapping.md and a full retrain.

# Three-class setup (changed 20 Oct 2026): Satire/Propaganda dropped because the
# only available source (TSHP-17) taught the model website style, not the class
# (Propaganda F1 0.99 on same-site validation, 0.08 on unseen-site test).
# For the old five-class setup use:
#   classes: [Fabricated, Misleading, Satire, Propaganda, Credible]
#   and set extra_classes.enabled: true in week1.yaml
classes: [Fabricated, Misleading, Credible]   # index order = model label ids

liar:
  # LIAR's six PolitiFact truth ratings -> XFakeNet classes
  # Keys are quoted on purpose: unquoted true/false become YAML booleans.
  "pants-fire":  Fabricated
  "false":       Fabricated
  "barely-true": Misleading
  "half-true":   Misleading
  "mostly-true": Credible
  "true":        Credible

# LIAR has no Satire or Propaganda signal. These two classes come from an
# external labelled source (default: TSHP-17, Rashkin et al. 2017).
tshp17:
  # TSHP-17 integer labels: 1=Satire 2=Hoax 3=Propaganda 4=Trusted
  1: Satire
  3: Propaganda
  # 2 (Hoax) and 4 (Trusted) are NOT used by default: they would add
  # long-article Fabricated/Credible examples whose style differs from LIAR.
  # Set use_hoax_trusted: true in configs/week1.yaml to include them.
  2: Fabricated
  4: Credible

fakenewsnet:
  # Binary only; used for C-score calibration/evaluation, not five-class training.
  "fake": Fabricated
  "real": Credible

In [ ]:
%%writefile src/xfakenet/__init__.py
# package marker

In [ ]:
%%writefile src/xfakenet/common.py
"""Shared helpers: config loading, label mapping, seeding, text cleanup."""
from __future__ import annotations

import json
import random
import re
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[2]


def load_yaml(path: str | Path) -> dict:
    with open(ROOT / path if not Path(path).is_absolute() else path) as f:
        return yaml.safe_load(f)


def load_config() -> dict:
    return load_yaml("configs/week1.yaml")


def load_mapping() -> dict:
    return load_yaml("configs/label_mapping.yaml")


def class_names() -> list[str]:
    return load_mapping()["classes"]


def label2id() -> dict[str, int]:
    return {c: i for i, c in enumerate(class_names())}


def p(rel: str | Path) -> Path:
    """Resolve a repo-relative path."""
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def set_seed(seed: int) -> None:
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


_WS = re.compile(r"\s+")
_SENT = re.compile(r"(?<=[.!?])\s+")


def clean_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.replace("“", '"').replace("”", '"').replace("’", "'")
    return _WS.sub(" ", s).strip()


def truncate_words(text: str, max_words: int) -> str:
    """Keep whole leading sentences up to max_words; hard-cut if the first sentence is longer."""
    text = clean_text(text)
    out, n = [], 0
    for sent in _SENT.split(text):
        w = sent.split()
        if n + len(w) > max_words:
            if not out:
                out.append(" ".join(w[:max_words]))
            break
        out.append(sent)
        n += len(w)
    return " ".join(out)


def write_json(obj, path: str | Path) -> None:
    with open(p(path), "w") as f:
        json.dump(obj, f, indent=2, default=str)

In [ ]:
%%writefile src/xfakenet/segments.py
"""Split a statement into segments (sentences and clauses) for the sentence-attention layer.

About 81% of LIAR statements are a single sentence, so splitting only at sentence
boundaries would give the attention layer nothing to choose between. We also split
at ; : commas and dashes, then merge pieces shorter than `min_words` into their
neighbour so a segment is always a meaningful clause.
"""
from __future__ import annotations

import re

_SPLIT = re.compile(r"(?<=[.!?;:])\s+|,\s+|\s+--?\s+")


def segment_spans(text: str, max_segments: int = 8, min_words: int = 3) -> list[tuple[int, int]]:
    """Return character spans (start, end) of the segments of `text`."""
    spans, start = [], 0
    for m in _SPLIT.finditer(text):
        if m.start() > start:
            spans.append((start, m.start()))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    if not spans:
        return [(0, len(text))]

    words = lambda s: len(text[s[0]:s[1]].split())  # noqa: E731
    merged: list[tuple[int, int]] = []
    for s in spans:
        if merged and (words(s) < min_words or words(merged[-1]) < min_words):
            merged[-1] = (merged[-1][0], s[1])
        else:
            merged.append(s)
    if len(merged) > max_segments:
        merged = merged[: max_segments - 1] + [(merged[max_segments - 1][0], merged[-1][1])]
    return merged


def segments(text: str, max_segments: int = 8, min_words: int = 3) -> list[str]:
    return [text[a:b] for a, b in segment_spans(text, max_segments, min_words)]

In [ ]:
%%writefile src/xfakenet/explain_utils.py
"""Word- and segment-level explanation from attention weights (no torch needed)."""
from __future__ import annotations

from bisect import bisect_right

import numpy as np


def word_explanation(text, offsets, seg_ids, tok_att, seg_att, spans, probs, classes, thr) -> dict:
    """Turn token/segment attention into word-level and segment-level explanations (pure numpy)."""
    words, w_imp, pos = [], [], 0
    for w in text.split():
        a = text.index(w, pos)
        words.append((a, a + len(w), w))
        w_imp.append(0.0)
        pos = a + len(w)
    starts = [a for a, _, _ in words]
    for j, (a, z) in enumerate(offsets):
        if z > a and seg_ids[j] >= 0 and words:
            k = max(0, bisect_right(starts, a) - 1)
            w_imp[k] += float(tok_att[j])      # a word's importance = sum over its sub-word tokens
    m = max(w_imp) if w_imp and max(w_imp) > 0 else 1.0
    norm = [x / m for x in w_imp]
    segs = [{"text": text[a:z], "weight": round(float(seg_att[k]), 4)}
            for k, (a, z) in enumerate(spans) if k < len(seg_att)]
    top = int(np.argmax([s["weight"] for s in segs])) if segs else 0
    return {
        "text": text,
        "pred": classes[int(np.argmax(probs))],
        "probs": {c: round(float(p_), 4) for c, p_ in zip(classes, probs)},
        "words": [{"w": w, "weight": round(x, 4)} for (_, _, w), x in zip(words, norm)],
        "highlighted": [w for (_, _, w), x in zip(words, norm) if x >= thr],
        "segments": segs,
        "top_segment": segs[top]["text"] if segs else text,
    }

In [ ]:
%%writefile src/xfakenet/model.py
"""XFakeNet classifier: RoBERTa + jointly trained token- and segment-attention (Module 1 + 3).

    tokens --RoBERTa--> h_t
    token attention (within each segment):   a_t = softmax_t( v_t . tanh(W_t h_t) )
    segment vector:                          s_k = sum_t a_t h_t
    segment attention:                       alpha_k = softmax_k( v_s . tanh(W_s s_k) )
    statement vector:                        d = sum_k alpha_k s_k
    prediction:                              softmax(W_c d)

The classifier only sees d, so the attention weights are part of the decision
itself (explanation produced in the forward pass), not computed afterwards.
Token importance shown to the user = a_t * alpha_(segment of t).
"""
from __future__ import annotations

import json
from bisect import bisect_right
from pathlib import Path
from typing import Sequence

import numpy as np
import torch
from torch import nn

from .explain_utils import word_explanation
from .segments import segment_spans


class XFakeNetModel(nn.Module):
    def __init__(self, encoder: nn.Module, num_labels: int, max_segments: int = 8,
                 att_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.encoder = encoder
        d = encoder.config.hidden_size
        self.K = max_segments
        self.tok_proj, self.tok_v = nn.Linear(d, att_dim), nn.Linear(att_dim, 1, bias=False)
        self.seg_proj, self.seg_v = nn.Linear(d, att_dim), nn.Linear(att_dim, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.cls = nn.Linear(d, num_labels)

    def forward(self, input_ids, attention_mask, seg_ids):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # B L D
        b, l, _ = h.shape
        e = self.tok_v(torch.tanh(self.tok_proj(h))).squeeze(-1).float()                       # B L
        ks = torch.arange(self.K, device=h.device)
        member = seg_ids.unsqueeze(1) == ks.view(1, -1, 1)                                       # B K L
        scores = e.unsqueeze(1).expand(b, self.K, l).masked_fill(~member, -1e4)
        a = torch.softmax(scores, -1) * member                                                   # B K L (empty segment -> 0)
        s = torch.bmm(a.to(h.dtype), h)                                                          # B K D
        valid = member.any(-1)                                                                   # B K
        u = self.seg_v(torch.tanh(self.seg_proj(s))).squeeze(-1).float().masked_fill(~valid, -1e4)
        alpha = torch.softmax(u, -1) * valid                                                     # B K
        doc = torch.bmm(alpha.unsqueeze(1).to(s.dtype), s).squeeze(1)                            # B D
        logits = self.cls(self.drop(doc)).float()
        token_imp = (a * alpha.unsqueeze(-1)).sum(1)                                             # B L
        return {"logits": logits, "token_att": token_imp, "seg_att": alpha}


def encode(tok, texts: Sequence[str], max_len: int, max_segments: int, device=None):
    """Tokenise and assign every token to a segment (special/pad tokens get -1)."""
    enc = tok(list(texts), truncation=True, max_length=max_len, padding=True,
              return_offsets_mapping=True, return_tensors="pt")
    offsets = enc.pop("offset_mapping")
    seg_ids = torch.full(enc["input_ids"].shape, -1, dtype=torch.long)
    spans_all = []
    for i, text in enumerate(texts):
        spans = segment_spans(text, max_segments)
        starts = [a for a, _ in spans]
        spans_all.append(spans)
        for j, (a, z) in enumerate(offsets[i].tolist()):
            if z > a:
                seg_ids[i, j] = max(0, bisect_right(starts, a) - 1)
    enc["seg_ids"] = seg_ids
    if device is not None:
        enc = {k: v.to(device) for k, v in enc.items()}
    return enc, offsets, spans_all


def build(encoder_name: str, num_labels: int, max_segments: int, att_dim: int, dropout: float = 0.1) -> XFakeNetModel:
    from transformers import AutoModel
    return XFakeNetModel(AutoModel.from_pretrained(encoder_name), num_labels, max_segments, att_dim, dropout)


def save(model: XFakeNetModel, tok, out_dir: str | Path, meta: dict) -> None:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out / "xfakenet.pt")
    model.encoder.config.save_pretrained(out / "encoder_config")
    tok.save_pretrained(out)
    json.dump(meta, open(out / "xfakenet_meta.json", "w"), indent=2)


def load(model_dir: str | Path, device: str = "cpu"):
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    d = Path(model_dir)
    meta = json.load(open(d / "xfakenet_meta.json"))
    enc = AutoModel.from_config(AutoConfig.from_pretrained(d / "encoder_config"))
    model = XFakeNetModel(enc, len(meta["classes"]), meta["max_segments"], meta["att_dim"])
    model.load_state_dict(torch.load(d / "xfakenet.pt", map_location="cpu"))
    return model.to(device).eval(), AutoTokenizer.from_pretrained(d), meta


class Predictor:
    """Batch prediction + explanations for a saved XFakeNet model."""

    def __init__(self, model_dir: str | Path, batch_size: int = 64, device: str | None = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model, self.tok, self.meta = load(model_dir, self.device)
        self.bs = batch_size

    @torch.no_grad()
    def _run(self, texts: Sequence[str]):
        for s in range(0, len(texts), self.bs):
            batch = list(texts[s:s + self.bs])
            enc, offsets, spans = encode(self.tok, batch, self.meta["max_len"], self.meta["max_segments"],
                                         self.device)
            with torch.autocast(device_type="cuda", enabled=self.device == "cuda"):
                out = self.model(**enc)
            yield batch, enc, offsets, spans, out

    def predict_proba(self, texts: Sequence[str]) -> np.ndarray:
        res = [torch.softmax(o["logits"], -1).cpu().numpy() for *_, o in self._run(texts)]
        return np.concatenate(res) if res else np.zeros((0, len(self.meta["classes"])))

    def explain(self, texts: Sequence[str], threshold: float | None = None) -> list[dict]:
        thr = self.meta.get("highlight_threshold", 0.65) if threshold is None else threshold
        out = []
        for batch, enc, offsets, spans, o in self._run(texts):
            probs = torch.softmax(o["logits"], -1).cpu().numpy()
            tok_att = o["token_att"].float().cpu().numpy()
            seg_att = o["seg_att"].float().cpu().numpy()
            seg_ids = enc["seg_ids"].cpu().numpy()
            offs = offsets.tolist()
            for i, text in enumerate(batch):
                out.append(word_explanation(text, offs[i], seg_ids[i], tok_att[i], seg_att[i], spans[i],
                                            probs[i], self.meta["classes"], thr))
        return out

In [ ]:
%%writefile src/xfakenet/textfooler.py
"""TextFooler (Jin et al., AAAI 2020, "Is BERT Really Robust?") without the TextAttack dependency.

Algorithm, as in the paper:
1. Rank words by importance: how much the true-class probability drops when the word is deleted.
2. For each word in that order, take its nearest neighbours in counter-fitted word
   embeddings (synonyms, cos >= 0.5).
3. Keep candidates that preserve the part of speech and keep the sentence meaning
   (sentence-embedding cosine >= threshold to the ORIGINAL text).
4. If a candidate flips the prediction, return the one closest in meaning (success).
   Otherwise keep the candidate that lowers the true-class probability most and continue.

Differences from the paper (documented for the report):
- Sentence similarity uses all-MiniLM-L6-v2 instead of the Universal Sentence Encoder.
- Model queries are batched and constraints are checked lazily (only on candidates
  the model output makes useful), which gives the same result order much faster.
- At most `max_change` (default 25%) of the words may be replaced.
- v2 quality filters (added after inspecting v1 output, where 28% of examples swapped a
  person/party/programme name and so changed the factual claim):
    * proper nouns, numbers and acronyms are never replaced (POS tags NNP/NNPS/CD, any
      capitalised word that is not sentence-initial, all-caps tokens);
    * candidates that are WordNet antonyms of the original word are rejected;
    * candidates on an offensive-word blocklist are rejected.

The class only needs four callables, so it can be unit-tested without a GPU.
"""
from __future__ import annotations

import math
import re
from typing import Callable, Sequence

import numpy as np

# Stopwords are never replaced (subset of the list used by TextFooler).
STOP = set("""a about above across after afterwards again against ain all almost alone along already also
although am among amongst an and another any anyhow anyone anything anyway anywhere are aren aren't around as
at back be became because become becomes becoming been before beforehand behind being below beside besides
between beyond both but by can cannot could couldn couldn't d didn didn't doesn doesn't don don't down due during
either else elsewhere empty enough even ever everyone everything everywhere except first for former formerly from
hadn hadn't hasn hasn't haven haven't he hence her here hereafter hereby herein hereupon hers herself him himself
his how however hundred i if in indeed into is isn isn't it it's its itself just latter latterly least ll may me
meanwhile mightn mightn't mine more moreover most mostly must mustn mustn't my myself namely needn needn't neither
never nevertheless next no nobody none noone nor not nothing now nowhere o of off on once one only onto or other
others otherwise our ours ourselves out over per please s same shan shan't she she's should've shouldn shouldn't
somehow something sometime somewhere such t than that that'll the their theirs them themselves then thence there
thereafter thereby therefore therein thereupon these they this those through throughout thru thus to too toward
towards under unless until up upon used ve was wasn wasn't we were weren weren't what whatever when whence
whenever where whereafter whereas whereby wherein whereupon wherever whether which while whither who whoever whole
whom whose why with within without won won't would wouldn wouldn't y yet you you'd you'll you're you've your yours
yourself yourselves says said""".split())

_AFFIX = re.compile(r"^(\W*)(.*?)(\W*)$")

# Candidates never used as replacements (offensive words that appear in embedding neighbourhoods).
BLOCKLIST = set("""slut sluts whore whores bitch bitches bastard bastards fuck fucking fucked shit shitty
crap cunt dick dicks cock pussy asshole arse arsehole retard retarded faggot fag nigger nigga
negro spic chink kike wetback tranny dyke hooker prostitute rapist""".split())


def split_tok(tok: str) -> tuple[str, str, str]:
    m = _AFFIX.match(tok)
    return m.group(1), m.group(2), m.group(3)


def match_case(src: str, cand: str) -> str:
    if src.isupper() and len(src) > 1:
        return cand.upper()
    if src[:1].isupper():
        return cand[:1].upper() + cand[1:]
    return cand


class TextFooler:
    def __init__(self,
                 predict: Callable[[Sequence[str]], np.ndarray],          # texts -> probs [n, C]
                 neighbours: Callable[[str], list[str]],                   # word -> synonyms
                 similarity: Callable[[str, Sequence[str]], np.ndarray],   # orig, texts -> cos [n]
                 pos_ok: Callable[[list[str], int, str], bool],            # words, idx, cand -> ok
                 protect: Callable[[list[str]], set] | None = None,        # words -> indices never replaced
                 banned: Callable[[str, str], bool] | None = None,         # (orig word, cand) -> reject?
                 sim_threshold: float = 0.80,
                 max_change: float = 0.25,
                 query_budget: int = 2000,
                 max_checks: int = 8):
        self.predict, self.neighbours = predict, neighbours
        self.similarity, self.pos_ok = similarity, pos_ok
        self.sim_threshold, self.max_change = sim_threshold, max_change
        self.query_budget, self.max_checks = query_budget, max_checks
        self.protect, self.banned = protect, banned

    def _ok(self, orig: str, words: list[str], i: int, cand_core: str, text: str) -> tuple[bool, float]:
        if not self.pos_ok(words, i, cand_core):
            return False, 0.0
        s = float(self.similarity(orig, [text])[0])
        return s >= self.sim_threshold, s

    def attack(self, text: str, label: int) -> dict:
        toks = text.split()
        parts = [split_tok(t) for t in toks]
        cores = [c for _, c, _ in parts]
        p0 = self.predict([text])[0]
        q = 1
        if int(np.argmax(p0)) != label:
            return {"status": "skipped", "queries": q}

        protected = set(self.protect(cores)) if self.protect else set()
        elig = [i for i, c in enumerate(cores)
                if c.isalpha() and len(c) >= 3 and c.lower() not in STOP
                and i not in protected
                and not (i > 0 and c[:1].isupper())        # capitalised mid-sentence: a name
                and not c.isupper()]                       # acronyms (CDC, NATO)
        if not elig:
            return {"status": "failed", "queries": q, "reason": "no_eligible_words"}

        # 1. word importance by deletion
        dels = [" ".join(toks[:i] + toks[i + 1:]) for i in elig]
        pd_ = self.predict(dels)
        q += len(dels)
        imp = p0[label] - pd_[:, label]
        flip = pd_.argmax(1) != label
        imp = imp + flip * (pd_.max(1) - p0[pd_.argmax(1)])   # paper's extra term when deletion flips
        order = [elig[k] for k in np.argsort(-imp)]

        cur = list(toks)
        cur_p = float(p0[label])
        changed, max_changes = 0, max(1, math.ceil(self.max_change * len(toks)))

        for i in order:
            if changed >= max_changes or q >= self.query_budget:
                break
            pre, core, post = parts[i]
            syns = [s for s in self.neighbours(core.lower())
                    if s.lower() != core.lower() and s.lower() not in BLOCKLIST
                    and not (self.banned and self.banned(core.lower(), s.lower()))]
            if not syns:
                continue
            cands = [match_case(core, s) for s in syns]
            texts = []
            for c in cands:
                t = list(cur)
                t[i] = pre + c + post
                texts.append(" ".join(t))
            pr = self.predict(texts)
            q += len(texts)
            words_now = [split_tok(t)[1] for t in cur]

            flipped = np.where(pr.argmax(1) != label)[0]
            if len(flipped):
                sims = self.similarity(text, [texts[k] for k in flipped])
                for k_idx in np.argsort(-sims):
                    k = flipped[k_idx]
                    if sims[k_idx] >= self.sim_threshold and self.pos_ok(words_now, i, cands[k]):
                        out = texts[k]
                        return {"status": "success", "text": out, "queries": q,
                                "words_changed": changed + 1,
                                "similarity": round(float(sims[k_idx]), 4),
                                "new_label_id": int(pr[k].argmax()),
                                "orig_conf": round(float(p0[label]), 4),
                                "adv_conf": round(float(pr[k][label]), 4)}

            # no valid flip: keep the best probability-lowering candidate that passes the constraints
            for k in np.argsort(pr[:, label])[: self.max_checks]:
                if pr[k, label] >= cur_p:
                    break
                ok, _ = self._ok(text, words_now, i, cands[k], texts[k])
                if ok:
                    cur[i] = pre + cands[k] + post
                    cur_p = float(pr[k, label])
                    changed += 1
                    break
        return {"status": "failed", "queries": q, "words_changed": changed}


# ---------- real components (used on Colab / the GPU server) ----------

class CounterFitted:
    """Nearest neighbours in counter-fitted vectors (Mrksic et al., 2016), as used by TextFooler."""

    def __init__(self, path: str, k: int = 50, min_cos: float = 0.5):
        words, vecs = [], []
        opener = open
        if path.endswith(".zip"):
            import io
            import zipfile
            z = zipfile.ZipFile(path)
            name = [n for n in z.namelist() if n.endswith(".txt")][0]
            fh = io.TextIOWrapper(z.open(name), encoding="utf-8")
        else:
            fh = opener(path, encoding="utf-8")
        for line in fh:
            w, *v = line.rstrip().split(" ")
            if len(v) < 10:
                continue
            words.append(w)
            vecs.append(np.asarray(v, dtype=np.float32))
        m = np.vstack(vecs)
        self.M = m / np.linalg.norm(m, axis=1, keepdims=True).clip(1e-8)
        self.words, self.idx = words, {w: i for i, w in enumerate(words)}
        self.k, self.min_cos, self.cache = k, min_cos, {}

    def __call__(self, w: str) -> list[str]:
        if w in self.cache:
            return self.cache[w]
        i = self.idx.get(w)
        if i is None:
            self.cache[w] = []
            return []
        sims = self.M @ self.M[i]
        kk = min(self.k + 1, len(sims) - 1)
        top = np.argpartition(-sims, kk)[: kk + 1]
        top = top[np.argsort(-sims[top])]
        res = [self.words[j] for j in top if j != i and sims[j] >= self.min_cos][: self.k]
        self.cache[w] = res
        return res


class WordNetSyn:
    """Fallback synonym source if the counter-fitted vectors cannot be downloaded."""

    def __init__(self, k: int = 50):
        from nltk.corpus import wordnet as wn
        self.wn, self.k, self.cache = wn, k, {}

    def __call__(self, w: str) -> list[str]:
        if w not in self.cache:
            out = []
            for s in self.wn.synsets(w):
                for l in s.lemma_names():
                    l = l.replace("_", " ")
                    if " " not in l and l.lower() != w and l not in out:
                        out.append(l)
            self.cache[w] = out[: self.k]
        return self.cache[w]


def make_pos_ok(window: int = 3):
    import nltk

    def coarse(tag: str) -> str:
        return tag[0] if tag[0] in "NVJR" else tag

    def pos_ok(words: list[str], i: int, cand: str) -> bool:
        lo, hi = max(0, i - window), min(len(words), i + window + 1)
        a = [w if w else "," for w in words[lo:hi]]
        b = list(a)
        b[i - lo] = cand
        ta, tb = nltk.pos_tag(a), nltk.pos_tag(b)
        return coarse(ta[i - lo][1]) == coarse(tb[i - lo][1])

    return pos_ok


def make_protect():
    """Indices of proper nouns and numbers (never replaced)."""
    import nltk

    def protect(words: list[str]) -> set:
        tags = nltk.pos_tag([w if w else "," for w in words])
        return {i for i, (_, t) in enumerate(tags) if t in ("NNP", "NNPS", "CD")}

    return protect


def make_banned():
    """Reject WordNet antonyms of the original word (e.g. died -> lives)."""
    from nltk.corpus import wordnet as wn
    cache: dict[str, set] = {}

    def antonyms(w: str) -> set:
        if w not in cache:
            out = set()
            for syn in wn.synsets(w):
                for lem in syn.lemmas():
                    for a in lem.antonyms():
                        out.add(a.name().lower())
                        for rel in a.derivationally_related_forms():
                            out.add(rel.name().lower())
            cache[w] = out
        return cache[w]

    def banned(orig: str, cand: str) -> bool:
        if cand in antonyms(orig) or orig in antonyms(cand):
            return True
        # also compare base forms (died/die vs lives/live)
        base_o, base_c = wn.morphy(orig) or orig, wn.morphy(cand) or cand
        return base_c in antonyms(base_o) or base_o in antonyms(base_c)

    return banned


def make_similarity(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer(model_name)
    cache: dict[str, np.ndarray] = {}

    def sim(orig: str, texts: Sequence[str]) -> np.ndarray:
        if orig not in cache:
            if len(cache) > 50:
                cache.clear()
            cache[orig] = st.encode([orig], normalize_embeddings=True)[0]
        e = st.encode(list(texts), normalize_embeddings=True, batch_size=64)
        return e @ cache[orig]

    return sim


def make_predict(model_dir: str, max_len: int = 64, batch_size: int = 64):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(dev).eval()

    @torch.no_grad()
    def predict(texts: Sequence[str]) -> np.ndarray:
        out = []
        for s in range(0, len(texts), batch_size):
            enc = tok(list(texts[s:s + batch_size]), truncation=True, max_length=max_len,
                      padding=True, return_tensors="pt").to(dev)
            with torch.autocast(device_type="cuda", enabled=dev == "cuda"):
                logits = model(**enc).logits
            out.append(torch.softmax(logits.float(), -1).cpu().numpy())
        return np.concatenate(out)

    return predict

In [ ]:
%%writefile scripts/07_textfooler_generate.py
"""Generate TextFooler adversarial examples against the week-1 baseline.

Uses src/xfakenet/textfooler.py (the Jin et al. algorithm, no TextAttack needed).

--split train : attacks TRAIN statements until the number of SUCCESSFUL examples
                reaches 15% of the final corpus (70/15/15), with per-class quotas.
                These are the augmentation data for week 3 (gold label kept).
--split test  : attacks a fixed stratified sample of the TEST split. The attack
                success rate is the robustness number for week 4. Never trained on.

Resumable: ids already in the output file are skipped and existing successes count.

Usage: python scripts/07_textfooler_generate.py --split train
       python scripts/07_textfooler_generate.py --split test
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import urllib.request
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed  # noqa: E402
from xfakenet import textfooler as tf  # noqa: E402

TF_VERSION = 2   # v2: names/numbers protected, antonyms + offensive words blocked


def synonym_source(cfg: dict):
    import zipfile
    tc = cfg["textfooler"]
    path = p(tc["embeddings_path"])
    urls = [tc["embeddings_url"],
            "https://media.githubusercontent.com/media/nmrksic/counter-fitting/master/word_vectors/counter-fitted-vectors.txt.zip"]
    for url in urls:
        if path.exists() and zipfile.is_zipfile(path):
            break
        if path.exists():
            os.remove(path)
        print(f"[textfooler] downloading counter-fitted vectors from {url} ...", flush=True)
        try:
            urllib.request.urlretrieve(url, path)
        except Exception as e:
            print(f"[textfooler] download failed ({e})", flush=True)
    if path.exists() and not zipfile.is_zipfile(path):
        os.remove(path)
        print("[textfooler] download was not a valid zip; using WordNet synonyms instead", flush=True)
    if path.exists():
        print("[textfooler] synonyms: counter-fitted vectors", flush=True)
        return tf.CounterFitted(str(path), k=tc["k"], min_cos=0.5), "counter-fitted"
    import nltk
    nltk.download("wordnet", quiet=True)
    return tf.WordNetSyn(k=tc["k"]), "wordnet"


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="train", choices=["train", "test"])
    ap.add_argument("--n", type=int, default=None)
    ap.add_argument("--victim", default=None)
    ap.add_argument("--victim-kind", default="hf", choices=["hf", "xfakenet"],
                    help="hf = Hugging Face classifier (baseline); xfakenet = XFakeNet model (week 3)")
    ap.add_argument("--out", default=None, help="override output file")
    args = ap.parse_args()

    cfg = load_config()
    tc = cfg["textfooler"]
    set_seed(cfg["seed"])

    import nltk
    for r in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger", "wordnet"]:
        nltk.download(r, quiet=True)

    syn, syn_name = synonym_source(cfg)
    victim = str(p(args.victim or tc["victim_dir"]))
    if args.victim_kind == "xfakenet":
        from xfakenet.model import Predictor
        predict_fn = Predictor(victim).predict_proba
    else:
        predict_fn = tf.make_predict(victim, cfg["baseline"]["max_len"])
    attacker = tf.TextFooler(
        predict=predict_fn,
        neighbours=syn, similarity=tf.make_similarity(tc["sim_model"]), pos_ok=tf.make_pos_ok(),
        protect=tf.make_protect(), banned=tf.make_banned(),
        sim_threshold=tc["sim_threshold"], max_change=tc["max_change"], query_budget=tc["query_budget"])

    df = pd.read_csv(p(f"{cfg['paths']['processed']}/corpus_{args.split}.csv"))
    df["id"] = df["id"].astype(str)
    out = p(args.out) if args.out else p(tc["out_train"] if args.split == "train" else tc["out_test"])
    prev = [json.loads(line) for line in open(out)] if out.exists() else []
    if prev and prev[0].get("tf_version") != TF_VERSION:
        old = out.with_name(out.stem + ".v1_unfiltered.jsonl")
        out.rename(old)
        print(f"[textfooler] archived old unfiltered output to {old.name}; regenerating", flush=True)
        prev = []
    done = {r["id"] for r in prev}

    if args.split == "train":
        a = cfg["augmentation"]
        target = args.n or round(len(df) * a["textfooler_share"] / a["original_share"])
        frac = df["label"].value_counts(normalize=True)
        quota = {c: round(target * f) for c, f in frac.items()}
        have = pd.Series([r["label"] for r in prev if r["status"] == "success"]).value_counts().to_dict()
        order = df.sample(frac=1.0, random_state=cfg["seed"])        # whole train, shuffled
    else:
        target = args.n or tc["test_n"]
        frac = df["label"].value_counts(normalize=True)
        order = pd.concat([g.sample(min(len(g), round(target * frac[c])), random_state=cfg["seed"])
                           for c, g in df.groupby("label")]).sample(frac=1.0, random_state=cfg["seed"])
        quota, have = None, {}
    todo = order[~order["id"].isin(done)]
    print(f"[textfooler] victim={victim} split={args.split} synonyms={syn_name} target={target} "
          f"already_done={len(done)} successes_so_far={sum(have.values())}", flush=True)

    t0, n, counts = time.time(), 0, {}
    with open(out, "a") as f:
        for row in todo.itertuples(index=False):
            if quota is not None:
                if all(have.get(c, 0) >= q for c, q in quota.items()):
                    break
                if have.get(row.label, 0) >= quota[row.label]:
                    continue
            try:
                res = attacker.attack(row.text, int(row.label_id))
            except Exception as e:  # keep the long job alive
                res = {"status": "error", "error": repr(e)[:200]}
            rec = {"tf_version": TF_VERSION, "id": row.id, "label": row.label, "label_id": int(row.label_id),
                   "source": row.source, "original": row.text, "synonyms": syn_name, **res}
            f.write(json.dumps(rec) + "\n")
            f.flush()
            counts[res["status"]] = counts.get(res["status"], 0) + 1
            if res["status"] == "success":
                have[row.label] = have.get(row.label, 0) + 1
            n += 1
            if n % 50 == 0:
                rate = n / (time.time() - t0)
                print(f"[textfooler] {n} attacked | {counts} | successes {sum(have.values())}/{target} "
                      f"| {rate:.1f}/s", flush=True)
    print(f"[textfooler] finished {counts} | successes by class {have}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/11_train_xfakenet.py
"""Train the XFakeNet classifier (RoBERTa + token/segment attention), class-weighted CE.

--data orig : LIAR training split only            -> runs/xfakenet_noaug  (ablation: no augmentation)
--data aug  : 70% LIAR + 15% paraphrase + 15% TF  -> runs/xfakenet_v2     (robust model v2)

The best epoch is chosen by validation macro-F1. Validation and test are always the
clean LIAR splits.

Output: runs/<run>/xfakenet.pt, xfakenet_meta.json, tokenizer, metrics.json,
        test_predictions.csv, confusion_test.csv

Usage : python scripts/11_train_xfakenet.py --data aug --run-name xfakenet_v2
        python scripts/11_train_xfakenet.py --smoke        (1-minute check on a few rows)
"""
from __future__ import annotations

import argparse
import math
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet import model as xm  # noqa: E402
from xfakenet.common import class_names, load_config, p, set_seed, write_json  # noqa: E402


def report(df: pd.DataFrame, pred: np.ndarray, classes: list[str]) -> dict:
    y = df["label_id"].to_numpy()
    return {
        "n": len(df),
        "accuracy": round(accuracy_score(y, pred), 4),
        "macro_f1": round(f1_score(y, pred, average="macro", zero_division=0), 4),
        "per_class": classification_report(y, pred, labels=list(range(len(classes))),
                                           target_names=classes, output_dict=True, zero_division=0),
    }


@torch.no_grad()
def predict(model, tok, texts, cfg, device, bs=128):
    model.eval()
    out = []
    for s in range(0, len(texts), bs):
        enc, _, _ = xm.encode(tok, texts[s:s + bs], cfg["max_len"], cfg["max_segments"], device)
        with torch.autocast(device_type="cuda", enabled=device == "cuda"):
            out.append(model(**enc)["logits"].float().cpu())
    return torch.cat(out).numpy()


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="aug", choices=["orig", "aug"])
    ap.add_argument("--run-name", default=None)
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--out-dir", default=None, help="absolute output folder (e.g. Colab local disk instead of Drive)")
    ap.add_argument("--train-file", default=None, help="override training csv (e.g. round-2 corpus)")
    ap.add_argument("--lr", type=float, default=None, help="encoder learning rate")
    ap.add_argument("--head-lr", type=float, default=None, help="attention layers + classifier learning rate")
    ap.add_argument("--batch-size", type=int, default=None)
    ap.add_argument("--dropout", type=float, default=None)
    ap.add_argument("--skip-test", action="store_true", help="grid search: never touch the test set")
    args = ap.parse_args()

    cfg = load_config()
    if args.seed is not None:
        cfg["seed"] = args.seed
    xc = dict(cfg["xfakenet"])
    for key, val in [("lr", args.lr), ("head_lr", args.head_lr), ("batch_size", args.batch_size),
                     ("dropout", args.dropout)]:
        if val is not None:
            xc[key] = val
    xc.setdefault("dropout", 0.1)
    set_seed(cfg["seed"])
    classes = class_names()
    d = cfg["paths"]["processed"]
    run = args.run_name or ("xfakenet_v2" if args.data == "aug" else "xfakenet_noaug")
    if args.smoke:
        run = "xfakenet_smoke"
    out_dir = Path(args.out_dir) if args.out_dir else p(f"{cfg['paths']['runs']}/{run}")
    out_dir.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_file = args.train_file or (f"{d}/corpus_aug_train.csv" if args.data == "aug" else f"{d}/corpus_train.csv")
    train = pd.read_csv(p(train_file))
    valid = pd.read_csv(p(f"{d}/corpus_valid.csv"))
    test = pd.read_csv(p(f"{d}/corpus_test.csv"))
    epochs = args.epochs or xc["epochs"]
    if args.smoke:
        train, valid, test, epochs = train.sample(96, random_state=0), valid.head(64), test.head(64), 1
    if "source" not in train:
        train["source"] = "liar"
    print(f"[{run}] train={len(train)} {train['source'].value_counts().to_dict()} | device={device}", flush=True)

    counts = np.bincount(train["label_id"], minlength=len(classes)).astype(float)
    w = counts.sum() / (len(classes) * np.maximum(counts, 1))
    weights = torch.tensor(w / w.mean(), dtype=torch.float, device=device)

    from transformers import AutoTokenizer, get_linear_schedule_with_warmup
    tok = AutoTokenizer.from_pretrained(xc["encoder"])
    model = xm.build(xc["encoder"], len(classes), xc["max_segments"], xc["att_dim"], xc["dropout"]).to(device)

    head = [p_ for n, p_ in model.named_parameters() if not n.startswith("encoder.")]
    enc_params = [p_ for n, p_ in model.named_parameters() if n.startswith("encoder.")]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": xc["lr"]},
                             {"params": head, "lr": xc["head_lr"]}], weight_decay=xc["weight_decay"])
    steps = math.ceil(len(train) / xc["batch_size"]) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(steps * xc["warmup_ratio"]), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")

    texts, labels = train["text"].tolist(), train["label_id"].to_numpy()
    import tempfile
    best_f1, best_epoch, history = -1.0, -1, []
    tmp = Path(tempfile.gettempdir()) / f"{run}_best"   # local disk, not Drive (faster)
    rng = np.random.default_rng(cfg["seed"])
    for ep in range(1, epochs + 1):
        model.train()
        order, t0, run_loss = rng.permutation(len(train)), time.time(), 0.0
        for step, s in enumerate(range(0, len(order), xc["batch_size"]), 1):
            idx = order[s:s + xc["batch_size"]]
            enc, _, _ = xm.encode(tok, [texts[i] for i in idx], xc["max_len"], xc["max_segments"], device)
            y = torch.tensor(labels[idx], device=device)
            with torch.autocast(device_type="cuda", enabled=device == "cuda"):
                logits = model(**enc)["logits"]
            loss = torch.nn.functional.cross_entropy(logits.float(), y, weight=weights)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            run_loss += loss.item()
            if step % 100 == 0:
                print(f"[{run}] epoch {ep} step {step} loss {run_loss / step:.4f}", flush=True)
        vp = predict(model, tok, valid["text"].tolist(), xc, device).argmax(-1)
        vf1 = f1_score(valid["label_id"], vp, average="macro")
        print(f"[{run}] epoch {ep} done in {time.time() - t0:.0f}s | train loss {run_loss / step:.4f} "
              f"| valid macro-F1 {vf1:.4f}", flush=True)
        history.append({"epoch": ep, "train_loss": round(run_loss / step, 4), "valid_macro_f1": round(float(vf1), 4)})
        if vf1 > best_f1:
            best_f1, best_epoch = vf1, ep
            xm.save(model, tok, tmp, {})

    # reload the best epoch and evaluate
    model.load_state_dict(torch.load(tmp / "xfakenet.pt", map_location=device))
    meta = {"classes": classes, "encoder": xc["encoder"], "max_len": xc["max_len"],
            "max_segments": xc["max_segments"], "att_dim": xc["att_dim"],
            "highlight_threshold": xc["highlight_threshold"], "data": args.data, "best_epoch": best_epoch,
            "dropout": xc["dropout"]}
    xm.save(model, tok, out_dir, meta)
    shutil.rmtree(tmp, ignore_errors=True)

    results = {"run_name": run, "data": args.data, "seed": cfg["seed"], "train_file": train_file, "train_sources": train["source"].value_counts().to_dict(),
               "config": xc, "epochs": epochs, "best_epoch": best_epoch, "lr": xc["lr"], "history": history}
    for name, df in [("valid", valid)] + ([] if args.skip_test else [("test", test)]):
        pred = predict(model, tok, df["text"].tolist(), xc, device).argmax(-1)
        results[name] = report(df.reset_index(drop=True), pred, classes)
        print(f"[{run}] {name}: acc={results[name]['accuracy']} macro_f1={results[name]['macro_f1']}", flush=True)
        if name == "test":
            cm = confusion_matrix(df["label_id"], pred, labels=list(range(len(classes))))
            pd.DataFrame(cm, index=classes, columns=classes).to_csv(out_dir / "confusion_test.csv")
            pd.DataFrame({"id": df["id"], "text": df["text"], "label": df["label"],
                          "pred": [classes[i] for i in pred]}).to_csv(out_dir / "test_predictions.csv", index=False)
    write_json(results, out_dir / "metrics.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/20_grid_search.py
"""Grid search over XFakeNet training hyperparameters (augmented corpus), selected on VALIDATION only.

Grid (configs/week1.yaml -> grid): encoder learning rate x batch size x dropout, each run trained for
`max_epochs` epochs with validation macro-F1 recorded after every epoch, so the number of epochs is
tuned too (lrs x batch sizes x dropouts x epochs configurations in total).

The test set is never evaluated during the search (--skip-test). Weights are kept on Colab's local
disk and deleted after each run; only the results go to Drive, one file per run, so the search can
be stopped and resumed (finished runs are skipped).

Output: results/grid/runs/<config>.json, results/grid/grid_results.csv, results/grid/grid_summary.md,
        results/grid/grid_best.json, results/grid/grid_heatmap.png
Usage : python scripts/20_grid_search.py [--stage run|summary|all] [--local /content/grid]
"""
from __future__ import annotations

import argparse
import itertools
import json
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from xfakenet.common import load_config, p, write_json  # noqa: E402


def name_of(lr: float, bs: int, do: float) -> str:
    return f"lr{lr:g}_bs{bs}_do{do:g}"


def combos(gc: dict):
    return list(itertools.product(gc["lrs"], gc["batch_sizes"], gc["dropouts"]))


def run_all(cfg: dict, local: str) -> None:
    gc = cfg["grid"]
    out = p(f"{gc['out_dir']}/runs")
    out.mkdir(parents=True, exist_ok=True)
    todo = [(lr, bs, do) for lr, bs, do in combos(gc) if not (out / f"{name_of(lr, bs, do)}.json").exists()]
    print(f"[grid] {len(combos(gc))} configurations x {gc['max_epochs']} epochs; {len(todo)} still to run", flush=True)
    for k, (lr, bs, do) in enumerate(todo, 1):
        name = name_of(lr, bs, do)
        run_dir = Path(local) / name
        print(f"[grid] ({k}/{len(todo)}) {name}", flush=True)
        cmd = [sys.executable, str(ROOT / "scripts/11_train_xfakenet.py"), "--data", "aug",
               "--run-name", f"grid_{name}", "--epochs", str(gc["max_epochs"]), "--lr", str(lr),
               "--batch-size", str(bs), "--dropout", str(do), "--skip-test", "--out-dir", str(run_dir)]
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        tail = []
        for line in proc.stdout:
            tail = (tail + [line])[-30:]
            if " done in " in line or "Error" in line or "Traceback" in line:
                print("   " + line.rstrip(), flush=True)
        if proc.wait() != 0 or not (run_dir / "metrics.json").exists():
            print("[grid] run FAILED; last output:\n" + "".join(tail), flush=True)
            shutil.rmtree(run_dir, ignore_errors=True)
            continue
        m = json.load(open(run_dir / "metrics.json"))
        write_json({"config": name, "lr": lr, "batch_size": bs, "dropout": do,
                    "best_epoch": m["best_epoch"], "valid_macro_f1": m["valid"]["macro_f1"],
                    "valid_accuracy": m["valid"]["accuracy"], "history": m["history"]},
                   f"{gc['out_dir']}/runs/{name}.json")
        shutil.rmtree(run_dir, ignore_errors=True)


def summarise(cfg: dict) -> dict:
    gc = cfg["grid"]
    rows = [json.load(open(f)) for f in sorted(p(f"{gc['out_dir']}/runs").glob("*.json"))]
    if not rows:
        print("[grid] no finished runs yet")
        return {}
    df = pd.DataFrame(rows).sort_values("valid_macro_f1", ascending=False).reset_index(drop=True)
    df.drop(columns=["history"]).to_csv(p(f"{gc['out_dir']}/grid_results.csv"), index=False)
    best = df.iloc[0]
    epochs = pd.DataFrame([{"config": r["config"], **h} for r in rows for h in r["history"]])
    best_json = {"config": best["config"], "lr": float(best["lr"]), "batch_size": int(best["batch_size"]),
                 "dropout": float(best["dropout"]), "best_epoch": int(best["best_epoch"]),
                 "valid_macro_f1": float(best["valid_macro_f1"]), "n_runs": len(df),
                 "n_total": len(combos(gc))}
    write_json(best_json, f"{gc['out_dir']}/grid_best.json")

    L = [f"Grid search: {len(df)} of {len(combos(gc))} configurations finished, each trained up to "
         f"{gc['max_epochs']} epochs ({len(epochs)} configuration x epoch points). Selected on validation "
         f"macro-F1; the test set was not used.", "",
         "| Rank | Encoder LR | Batch size | Dropout | Best epoch | Valid macro-F1 | Valid accuracy |",
         "|---|---|---|---|---|---|---|"]
    for i, r in df.iterrows():
        L.append(f"| {i + 1} | {r['lr']:g} | {int(r['batch_size'])} | {r['dropout']:g} | {int(r['best_epoch'])} | "
                 f"{r['valid_macro_f1']:.4f} | {r['valid_accuracy']:.4f} |")
    L += ["", "Average best validation macro-F1 for each value (how much each setting matters):", "",
          "| Hyperparameter | Value | Mean valid macro-F1 | Runs |", "|---|---|---|---|"]
    for col, lab in [("lr", "Encoder LR"), ("batch_size", "Batch size"), ("dropout", "Dropout")]:
        for v, g in df.groupby(col):
            L.append(f"| {lab} | {v:g} | {g['valid_macro_f1'].mean():.4f} | {len(g)} |")
    ep = epochs.groupby("epoch")["valid_macro_f1"].agg(["mean", "max"])
    L += ["", "| Epoch | Mean valid macro-F1 over configs | Best over configs |", "|---|---|---|"] + \
         [f"| {e} | {r['mean']:.4f} | {r['max']:.4f} |" for e, r in ep.iterrows()]
    spread = df["valid_macro_f1"].max() - df["valid_macro_f1"].min()
    L += ["", f"Best: encoder LR {best['lr']:g}, batch size {int(best['batch_size'])}, dropout {best['dropout']:g}, "
          f"{int(best['best_epoch'])} epochs (valid macro-F1 {best['valid_macro_f1']:.4f}). Spread across the grid: "
          f"{spread:.4f}."]
    open(p(f"{gc['out_dir']}/grid_summary.md"), "w").write("\n".join(L))
    print("\n".join(L))

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        bss, lrs, dos = sorted(df["batch_size"].unique()), sorted(df["lr"].unique()), sorted(df["dropout"].unique())
        fig, axes = plt.subplots(1, len(bss), figsize=(4.6 * len(bss), 3.6), squeeze=False, constrained_layout=True)
        vmin, vmax = df["valid_macro_f1"].min(), df["valid_macro_f1"].max()
        for k, (ax, bs) in enumerate(zip(axes[0], bss)):
            grid = np.full((len(dos), len(lrs)), np.nan)
            for _, r in df[df.batch_size == bs].iterrows():
                grid[dos.index(r["dropout"]), lrs.index(r["lr"])] = r["valid_macro_f1"]
            im = ax.imshow(grid, cmap="Oranges", vmin=vmin, vmax=vmax, aspect="auto")
            for i in range(len(dos)):
                for j in range(len(lrs)):
                    if not np.isnan(grid[i, j]):
                        dark = (grid[i, j] - vmin) / max(vmax - vmin, 1e-9) > 0.6
                        ax.text(j, i, f"{grid[i, j]:.3f}", ha="center", va="center", fontsize=10,
                                color="white" if dark else "black")
            ax.set_xticks(range(len(lrs)), [f"{v:g}" for v in lrs])
            ax.set_yticks(range(len(dos)), [f"{v:g}" for v in dos])
            ax.set_xlabel("Encoder learning rate")
            if k == 0:
                ax.set_ylabel("Dropout")
            ax.set_title(f"Batch size {bs}")
        fig.suptitle("Validation macro-F1 across the grid (best epoch per run)", fontsize=12)
        fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8)
        fig.savefig(p(f"{gc['out_dir']}/grid_heatmap.png"), dpi=150, bbox_inches="tight")
    except Exception as e:
        print("[grid] heatmap skipped:", e)
    return best_json


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", default="all", choices=["all", "run", "summary"])
    ap.add_argument("--local", default="/content/grid", help="local folder for temporary weights")
    args = ap.parse_args()
    cfg = load_config()
    if args.stage in ("all", "run"):
        run_all(cfg, args.local)
    if args.stage in ("all", "summary"):
        summarise(cfg)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/21_final_tuned.py
"""Retrain the best grid-search configuration with 3 seeds and compare it with the default settings.

For each seed: train XFakeNet v2 (augmented corpus) with the chosen hyperparameters, evaluate on the
clean LIAR test set, then attack it with a fresh TextFooler run on the same 499 test statements used
for every other model. The seed-42 model is kept on Drive as runs/xfakenet_tuned.

Output: results/grid/final/seed<k>.json, data/augment/textfooler_test_xfakenet_tuned_s<k>.jsonl,
        results/grid/tuned_vs_default.md / .json
Usage : python scripts/21_final_tuned.py [--stage run|summary|all]
"""
from __future__ import annotations

import argparse
import json
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from xfakenet.common import load_config, p, write_json  # noqa: E402


def lines(f: Path) -> int:
    return sum(1 for _ in open(f)) if f.exists() else 0


def attack_stats(f: Path):
    if not f.exists():
        return None
    t = pd.DataFrame([json.loads(l) for l in open(f)])
    vc = t["status"].value_counts()
    s, fl, n = int(vc.get("success", 0)), int(vc.get("failed", 0)), len(t)
    return {"acc_under_attack": fl / n, "success_rate": s / max(1, s + fl)}


def sh(cmd: list[str], keep: tuple[str, ...]) -> int:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in proc.stdout:
        tail = (tail + [line])[-30:]
        if any(k in line for k in keep):
            print("   " + line.rstrip(), flush=True)
    rc = proc.wait()
    if rc != 0:
        print("".join(tail), flush=True)
    return rc


def run(cfg: dict, local: str) -> None:
    gc = cfg["grid"]
    best = json.load(open(p(f"{gc['out_dir']}/grid_best.json")))
    print(f"[tuned] best config {best['config']} (valid macro-F1 {best['valid_macro_f1']:.4f})", flush=True)
    for seed in gc["final_seeds"]:
        res_f = p(f"{gc['out_dir']}/final/seed{seed}.json")
        atk_f = p(f"data/augment/textfooler_test_xfakenet_tuned_s{seed}.jsonl")
        run_dir = Path(local) / f"tuned_s{seed}"
        if res_f.exists() and lines(atk_f) >= 499:
            print(f"[tuned] seed {seed}: done", flush=True)
            continue
        if not (run_dir / "xfakenet.pt").exists():
            print(f"[tuned] seed {seed}: training", flush=True)
            sh([sys.executable, str(ROOT / "scripts/11_train_xfakenet.py"), "--data", "aug",
                "--run-name", f"xfakenet_tuned_s{seed}", "--seed", str(seed), "--epochs", str(gc["max_epochs"]),
                "--lr", str(best["lr"]), "--batch-size", str(best["batch_size"]), "--dropout", str(best["dropout"]),
                "--out-dir", str(run_dir)], (" done in ", "test:", "Error", "Traceback"))
            if (run_dir / "metrics.json").exists():
                shutil.copy(run_dir / "metrics.json", res_f)
        if lines(atk_f) < 499 and (run_dir / "xfakenet.pt").exists():
            print(f"[tuned] seed {seed}: TextFooler attack", flush=True)
            sh([sys.executable, str(ROOT / "scripts/07_textfooler_generate.py"), "--split", "test",
                "--victim", str(run_dir), "--victim-kind", "xfakenet", "--out", str(atk_f)],
               ("[textfooler] finished", "Error", "Traceback"))
        if seed == 42 and (run_dir / "xfakenet.pt").exists() and not p("runs/xfakenet_tuned/xfakenet.pt").exists():
            shutil.copytree(run_dir, p("runs/xfakenet_tuned"), dirs_exist_ok=True)


def summarise(cfg: dict) -> None:
    gc = cfg["grid"]
    best = json.load(open(p(f"{gc['out_dir']}/grid_best.json")))
    groups = {
        "Default settings (XFakeNet v2)": (
            [p("runs/xfakenet_v2/metrics.json"), p("runs/xfakenet_v2_s1/metrics.json"), p("runs/xfakenet_v2_s2/metrics.json")],
            [p("data/augment/textfooler_test_xfakenet_v2.jsonl"), p("data/augment/textfooler_test_xfakenet_v2_s1.jsonl"),
             p("data/augment/textfooler_test_xfakenet_v2_s2.jsonl")]),
        "Tuned (grid-search best)": (
            [p(f"{gc['out_dir']}/final/seed{s}.json") for s in gc["final_seeds"]],
            [p(f"data/augment/textfooler_test_xfakenet_tuned_s{s}.jsonl") for s in gc["final_seeds"]]),
    }

    def ms(xs):
        xs = [x for x in xs if x is not None]
        if not xs:
            return "–"
        return f"{np.mean(xs):.3f}" if len(xs) == 1 else f"{np.mean(xs):.3f} ± {np.std(xs, ddof=1):.3f}"

    out, L = {}, []
    xc = cfg["xfakenet"]
    L += [f"Default settings: encoder LR {xc['lr']:g}, batch size {xc['batch_size']}, dropout 0.1, "
          f"{xc['epochs']} epochs. Tuned: encoder LR {best['lr']:g}, batch size {best['batch_size']}, "
          f"dropout {best['dropout']:g}, up to {gc['max_epochs']} epochs (best epoch by validation).", "",
          "| Model | Seeds | Valid macro-F1 | Test accuracy | Test macro-F1 | Accuracy under attack | Attack success rate |",
          "|---|---|---|---|---|---|---|"]
    for name, (mfiles, afiles) in groups.items():
        ms_ = [json.load(open(f)) for f in mfiles if f.exists()]
        at = [a for a in (attack_stats(f) for f in afiles) if a]
        if not ms_:
            continue
        row = {"valid_f1": [m["valid"]["macro_f1"] for m in ms_],
               "test_acc": [m["test"]["accuracy"] for m in ms_ if "test" in m],
               "test_f1": [m["test"]["macro_f1"] for m in ms_ if "test" in m],
               "acc_under_attack": [a["acc_under_attack"] for a in at],
               "success": [a["success_rate"] for a in at]}
        out[name] = row
        L.append(f"| {name} | {len(ms_)} | {ms(row['valid_f1'])} | {ms(row['test_acc'])} | {ms(row['test_f1'])} | "
                 f"{ms(row['acc_under_attack'])} | {ms(row['success'])} |")
    write_json(out, f"{gc['out_dir']}/tuned_vs_default.json")
    open(p(f"{gc['out_dir']}/tuned_vs_default.md"), "w").write("\n".join(L))
    print("\n".join(L))


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", default="all", choices=["all", "run", "summary"])
    ap.add_argument("--local", default="/content/tuned")
    args = ap.parse_args()
    cfg = load_config()
    p(f"{cfg['grid']['out_dir']}/final").mkdir(parents=True, exist_ok=True)
    if args.stage in ("all", "run"):
        run(cfg, args.local)
    if args.stage in ("all", "summary"):
        summarise(cfg)


if __name__ == "__main__":
    main()

## 3. Install packages (for the TextFooler attack)

In [ ]:
!pip install -q sentence-transformers nltk pyyaml

## A. Grid search (resumable)
Each run prints its validation macro-F1 after every epoch. Finished runs are skipped when you re-run.

In [ ]:
!python scripts/20_grid_search.py --stage run
!python scripts/20_grid_search.py --stage summary > /dev/null
from IPython.display import Markdown, Image, display
display(Image("results/grid/grid_heatmap.png", width=820))
display(Markdown(open("results/grid/grid_summary.md").read()))

## B. Best setting with 3 seeds, compared with the default settings

In [ ]:
import json
best = json.load(open("results/grid/grid_best.json"))
if best["n_runs"] < best["n_total"]:
    print(f"Only {best['n_runs']} of {best['n_total']} grid runs are finished. Run section A again first.")
else:
    !python scripts/21_final_tuned.py

## C. Results zip (saved to Drive and downloaded)

In [ ]:
import os, shutil
from IPython.display import Markdown, display
if os.path.exists("results/grid/tuned_vs_default.md"):
    display(Markdown(open("results/grid/tuned_vs_default.md").read()))
os.makedirs("/content/w5", exist_ok=True)
for f in ["grid_summary.md", "grid_results.csv", "grid_heatmap.png", "grid_best.json",
          "tuned_vs_default.md", "tuned_vs_default.json"]:
    if os.path.exists(f"results/grid/{f}"): shutil.copy(f"results/grid/{f}", f"/content/w5/{f}")
shutil.make_archive("/content/xfakenet_gridsearch_results", "zip", "/content/w5")
shutil.copy("/content/xfakenet_gridsearch_results.zip", "results/final/xfakenet_gridsearch_results.zip")
from google.colab import drive
drive.flush_and_unmount()
print("Saved to Google Drive: My Drive > xfakenet > results > final > xfakenet_gridsearch_results.zip")
from google.colab import files
try:
    files.download("/content/xfakenet_gridsearch_results.zip")
except Exception:
    print("Browser download failed; use the copy on Google Drive.")